In [1]:
%load_ext autoreload
# %autoreload 2wandb.login()

In [2]:
import h5py, os, tqdm, glob, wandb, yaml
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.49'
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import optax
from flax import nnx
import orbax.checkpoint as ocp

from jaxpm import camels, data, plotting, graph
from jaxpm.painting import cic_paint, cic_read
from jaxpm.nn import MLP, CNN, HybridNet, AttentionGNN

jax.devices("gpu")

[cuda(id=0)]

In [3]:
wandb.login()

ERROR:wandb.jupyter:Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: arne-thomsen (eth-cosmo). Use `wandb login --relogin` to force relogin


True

# functions

## data

In [4]:
def load_hpm_data(HPM, with_particle, with_field, x_labels, y_labels):
    X_particle, X_field, Y_particle, Y_field = None, None, None, None
    
    all_y_labels = ["P", "U", "T"]
    y_indices = [i for i, label in enumerate(all_y_labels) if label in y_labels]

    with h5py.File(HPM, "r") as f:
        if with_particle:
            all_x_labels = ["scales", "rho", "fscalar", "vel_disp", "vel_div"]
            x_indices = [i for i, label in enumerate(all_x_labels) if label in x_labels]
            y_indices = [i for i, label in enumerate(all_y_labels) if label in y_labels]

            X_particle = f["X_particle"][..., x_indices]
            Y_particle = f["Y_particle"][..., y_indices]

        if with_field:
            all_x_labels = ["rho", "fscalar", "vel_disp", "vel_div"]
            x_indices = [i for i, label in enumerate(all_x_labels) if label in x_labels]
            y_indices = [i for i, label in enumerate(all_y_labels) if label in y_labels]

            X_field = f["X_field"][..., x_indices]
            Y_field = f["Y_field"][..., y_indices]

    return X_particle, X_field, Y_particle, Y_field


## training

In [5]:
@nnx.jit
def train_step(model, optimizer, x, y):

    def loss_fn(model):
        y_pred = model(x, training=True)
        return jnp.mean((y - y_pred) **2)

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads)

    return loss

@nnx.jit
def vali_loop(model, X, Y):
    losses = []
    for i in range(scales.shape[0]):
        x, y = tuple(x[i] for x in X), Y[i]
        
        losses.append(jnp.mean((y - model(*x))**2))
    losses = jnp.mean(jnp.stack(losses))

    return losses

# # NOTE this is GD, not SGD. There's no batches, all particles in the snapshot are evaluated every step
# def train_model(
#     model, 
#     optimizer,
#     X,
#     Y,
#     X_vali,
#     Y_vali,
#     total_steps=3_000,
#     vali_every=100,
#     learning_rate=1e-3,
#     cosine_decay=True,
# ):
#     losses = []
#     vali_steps = []
#     vali_losses = []
#     vali_loss = np.inf
#     for i in (pbar := tqdm.tqdm(range(total_steps))):
#         # select single snapshot
#         # TODO
#         j = np.random.choice(np.arange(21))
#         x, y = tuple(x[j] for x in X), Y[j]
        
#         loss = train_step(model, optimizer, x, y)
#         losses.append(loss)

#     #     if (i % vali_every == 0) and (i != 0) or i == total_steps - 1:
#     #         vali_steps.append(i)
#     #         vali_loss = vali_loop(model, X_vali, Y_vali)
#     #         vali_losses.append(vali_loss)

#     #     pbar.set_description(f"train={loss:.4f}, vali={vali_loss:.4f}")

#     # fig, ax = plt.subplots()
#     # ax.plot(losses, label="training")
#     # ax.plot(vali_steps, vali_losses, label="validation")
#     # ax.legend(loc="upper right")
#     # ax.set(yscale="log")
    

In [12]:
def train_model(
    model, 
    optimizer,
    X,
    Y,
    batch_size=512,
    total_steps=1000,
):
    losses = []
    for i in (pbar := tqdm.tqdm(range(total_steps))):
        # TODO select single snapshot
        j = np.random.choice(np.arange(X.shape[0]))
        # select batch
        k = np.random.choice(np.arange(X.shape[2]), batch_size)
        x, y = X[j,:,k], Y[j,:,k]
        
        loss = train_step(model, optimizer, x, y)
        losses.append(loss)
        if i % 100 == 0:
            try:
                wandb.log({"train_loss": loss})
            except:
                pass


# Network selection

In [24]:
def wandb_main():
    wandb.init(project="HPM-sweep")
    vali_loss = train_mlp(wandb.config)
    wandb.log({"vali_loss": vali_loss})

def train_mlp(config, eps=1e-8):    
    X_particle, _, Y_particle, _ = load_hpm_data(
        "/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/merged_hpm_parts=64,mesh=64.h5", 
        with_particle=True, 
        with_field=False, 
        x_labels=config["x_labels"], 
        y_labels=config["y_labels"],
    )

    vali_split = int(0.8 * X_particle.shape[0])
    X, Y = X_particle[:vali_split], Y_particle[:vali_split]
    X_vali, Y_vali = X_particle[vali_split:], Y_particle[vali_split:]
    
    model = MLP(
        X.shape[-1], 
        Y.shape[-1], 
        config["d_hidden"], 
        config["n_hidden"],
        nnx.Rngs(0),
        config["dropout_rate"],
        config["activation"],
        config["norm_type"],
    )

    optimizer = optax.chain(
        optax.clip_by_global_norm(1),
        optax.adam(config["learning_rate"])
    )
    optimizer = nnx.Optimizer(model, optimizer)
    # if cosine_decay:
    #     learning_rate = optax.cosine_decay_schedule(
    #         init_value=learning_rate, 
    #         decay_steps=total_steps, 
    #         alpha=0.1
    #     )

    train_model(
        model,
        optimizer,
        X,
        Y,
        total_steps=config["n_steps"],
    )

    vali_loss = 0
    for x_vali, y_vali in tqdm.tqdm(zip(X_vali, Y_vali), total=X_vali.shape[0]):
        print(x_vali.shape)
        vali_loss += jnp.mean((model(x_vali) - y_vali)**2)
    vali_loss /= X_vali.shape[0]
    
    return vali_loss


In [25]:
single_config = {
    "d_hidden": 64,
    "n_hidden": 4,
    "dropout_rate": 0.1,
    "activation": "relu",
    # "activation": jax.nn.relu,
    # "norm_type": "layer",
    "norm_type": None,
    "x_labels": ["scale", "rho", "fscalar"],
    "y_labels": ["P"],
    "learning_rate": 1e-4,
    "n_steps": 1_000,
}

train_mlp(single_config)

KeyboardInterrupt: 

In [9]:
isinstance("relu", str)

True

In [22]:
file = "/cluster/home/athomsen/flatiron/repos/JaxPM/dev/hpm/offline_regression/sweep/mlp_config.yaml"
with open(file, "r") as file:
    config = yaml.safe_load(file)

In [23]:
sweep_id = wandb.sweep(sweep=config, project="HPM-sweep")
wandb.agent(sweep_id, function=wandb_main, count=2)

Create sweep with ID: bgn7c8ke
Sweep URL: https://wandb.ai/eth-cosmo/HPM-sweep/sweeps/bgn7c8ke


wandb: Agent Starting Run: etb2hytw with config:
wandb: 	activation: sigmoid
wandb: 	d_hidden: 256
wandb: 	dropout_rate: 0.3487454360632678
wandb: 	learning_rate: 3.647001355620341e-05
wandb: 	n_hidden: 1
wandb: 	n_steps: 1000
wandb: 	norm_type: layer
wandb: 	x_labels: ['scale', 'rho', 'fscalar']
wandb: 	y_labels: P
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: WARNING Ignored wandb.init() arg project when running a sweep.
cat: /sys/module/amdgpu/initstate: No such file or directory
ERROR:root:Driver not initialized (amdgpu not found in modules)


  0%|          | 0/6 [00:00<?, ?it/s]


train_loss,█▂▁▁▁▁▁▁▁▁
train_loss,0.88475


Run etb2hytw errored:
Traceback (most recent call last):
  File "/cluster/software/stacks/2024-05/python-cuda/3.11.6/lib/python3.11/site-packages/wandb/agents/pyagent.py", line 307, in _run_job
    self._function()
  File "/scratch/tmp.33141232.athomsen/ipykernel_950695/2771188131.py", line 3, in wandb_main
    vali_loss = train_mlp(wandb.config)
                ^^^^^^^^^^^^^^^^^^^^^^^
  File "/scratch/tmp.33141232.athomsen/ipykernel_950695/2771188131.py", line 52, in train_mlp
    vali_loss += jnp.mean((model(x_vali) - y_vali)**2)
                           ^^^^^^^^^^^^^
  File "/cluster/home/athomsen/flatiron/repos/JaxPM/jaxpm/nn.py", line 172, in __call__
    x = self.linear_in(x)
        ^^^^^^^^^^^^^^^^^
  File "/cluster/home/athomsen/flatiron/lib/python3.11/site-packages/flax/nnx/nn/linear.py", line 367, in __call__
    y = self.dot_general(
        ^^^^^^^^^^^^^^^^^
  File "/cluster/software/stacks/2024-05/python-cuda/3.11.6/lib/python3.11/site-packages/jax/_src/lax/lax.py", lin

wandb: Ctrl + C detected. Stopping sweep.
  0%|          | 1/1000 [00:02<34:13,  2.06s/it]wandb: WARNING Unable to render progress bar, see the user log for details
wandb: ERROR Problem finishing run
Exception in thread Thread-26 (_run_job):
Traceback (most recent call last):
  File "/cluster/software/stacks/2024-05/python-cuda/3.11.6/lib/python3.11/site-packages/tqdm/std.py", line 1191, in __iter__
    self.update(n - last_print_n)
  File "/cluster/software/stacks/2024-05/python-cuda/3.11.6/lib/python3.11/site-packages/tqdm/std.py", line 1242, in update
    self.refresh(lock_args=self.lock_args)
  File "/cluster/software/stacks/2024-05/python-cuda/3.11.6/lib/python3.11/site-packages/tqdm/std.py", line 1347, in refresh
    self.display()
  File "/cluster/software/stacks/2024-05/python-cuda/3.11.6/lib/python3.11/site-packages/tqdm/std.py", line 1495, in display
    self.sp(self.__str__() if msg is None else msg)
  File "/cluster/software/stacks/2024-05/python-cuda/3.11.6/lib/python3.11/

# OLD

In [18]:
X_particle, X_field, Y_particle, Y_field = load_hpm_data(
    "/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/merged_hpm_parts=64,mesh=64.h5", 
    with_particle=True, 
    with_field=False, 
    x_labels=["scales", "rho", "fscalar"], 
    y_labels=["P"],
)

In [22]:
X_particle.shape

(27, 34, 262144, 3)

In [25]:
X_particle[:int(0.8*X_particle.shape[0])].shape

(21, 34, 262144, 3)

In [26]:
X_particle[int(0.8*X_particle.shape[0]):].shape

(6, 34, 262144, 3)

In [6]:
HPM = "/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/merged_hpm_parts=64,mesh=64.h5"
with h5py.File(HPM, "r") as f:
    # X_particle = f["X_particle"][...,x_indices]
    print(f["X_particle"].shape)
    temp = f["X_particle"][:]


(27, 34, 262144, 5)


In [12]:
def train_mlp(x_labels, y_labels, eps=1e-8, include_scale=True):
    X, _, Y, _ = data.get_offline_regression_data(
        train_dict, x_labels=x_labels, y_labels=y_labels, standardize_input=False, include_scale=include_scale
    )
    X_vali, _, Y_vali, _ = data.get_offline_regression_data(
        test_dict, x_labels=x_labels, y_labels=y_labels, standardize_input=False, include_scale=include_scale
    )

    # scaler = StandardScaler3D().fit(Y)
    # scaler = TimeIndependentScaler3D().fit(Y)
    # Y = scaler.transform(Y)
    # Y_vali = scaler.transform(Y_vali)
    # print("with standard scaler")

    model = MLP(
        X.shape[-1], 
        Y.shape[-1], 
        64, 
        4,
        nnx.Rngs(0),
        0.0,
    )
    
    train_model(
        model,
        (X,),
        Y,
        (X_vali,),
        Y_vali,
    )

    pred = model(X)
    pred_vali = model(X_vali)

    with jax.default_device(jax.devices("cpu")[0]):
        plot_preds(pred, Y, pred_vali, Y_vali)
    
        # pred = scaler.inverse_transform(pred)
        # pred_vali = scaler.inverse_transform(pred_vali)

        i = -1
        P_pred = normalized_paint(train_dict["gas_poss"][i], jnp.squeeze(10**pred[i]) - eps)
        P_pred_vali = normalized_paint(test_dict["gas_poss"][i], jnp.squeeze(10**pred_vali[i]) - eps)
        
        P_camels = normalized_paint(train_dict["gas_poss"][i], train_dict["gas_Ps"][i])
        P_camels_vali = normalized_paint(test_dict["gas_poss"][i], test_dict["gas_Ps"][i])
        
        plot_comparison(P_camels, P_pred, pred_label="MLP", suptitle="training set")
        plot_comparison(P_camels_vali, P_pred_vali, pred_label="MLP", suptitle="validation set")
    
    return model


# training

In [9]:
parts_per_dim = 64
mesh_per_dim = parts_per_dim
mesh_shape = [mesh_per_dim] * 3

HPM = "/cluster/scratch/athomsen/CV/hpm.h5"

with h5py.File(HPM, "r") as f:
    # print(f.keys())
    X_particle = f["X_particle"][:]
    Y_particle = f["Y_particle"][:]

In [8]:
X_particle.shape

(27, 34, 262144, 5)

In [19]:
all_x_labels = ["scales", "rho", "fscalar", "vel_disp", "vel_div"]
all_y_labels = ["P", "U", "T"]

x_labels = ["scales", "vel_div"]
y_labels = ["U"]

x_indices = [i for i, label in enumerate(all_x_labels) if label in x_labels]
y_indices = [i for i, label in enumerate(all_y_labels) if label in y_labels]

print(x_indices)
print(y_indices)

HPM = "/cluster/scratch/athomsen/CV/hpm.h5"
with h5py.File(HPM, "r") as f:
    X_particle = f["X_particle"][...,x_indices]
    Y_particle = f["Y_particle"][...,y_indices]


[0, 4]
[1]


In [23]:
jnp.array(x_indices) - 1

Array([-1,  3], dtype=int32)

In [ ]:
    HPM = "/cluster/scratch/athomsen/CV/hpm.h5"
    with h5py.File(HPM, "r") as f:
        X_particle = f["X_particle"][:]
        Y_particle = f["Y_particle"][:]


In [17]:
i_labels

[0, 4]

In [10]:
Y_particle.shape

(27, 34, 262144, 3)

In [13]:
x_labels = ["rho", "fscalar", "vel_disp", "vel_div"]
# x_labels = ["rho", "fscalar"]
y_labels = ["P"]

# model = train_mlp(x_labels, y_labels)
# # model = train_gnn(x_labels, y_labels)

# model = train_mlp(x_labels, y_labels, include_scale=False)
# # model = train_gnn(x_labels, y_labels)

In [14]:
# X, _, Y, _ = data.get_offline_regression_data(
#     train_dict, x_labels=x_labels, y_labels=y_labels, standardize_input=False, include_scale=True
# )

# # print(X.shape)
# # print(Y.shape)

In [ ]:
model = train_mlp(x_labels, y_labels)

train=0.2018, vali=0.5054:  28%|██▊       | 852/3000 [00:11<00:21, 99.79it/s] 

In [ ]:
# model = train_mlp(x_labels, y_labels, include_scale=False)

In [ ]:
# model = train_mlp(x_labels, y_labels, include_scale=False)

In [ ]:
# see https://flax.readthedocs.io/en/latest/guides/checkpointing.html
# checkpoint_file = os.path.join(os.getcwd(), "checkpoints/mlp_table_v1.jx")
# checkpoint_file = os.path.join(os.getcwd(), "checkpoints/mlp_table_illustris_cv1.jx")
# checkpoint_file = os.path.join(os.getcwd(), "checkpoints/mlp_table_v1_late_time.jx")
# checkpoint_file = os.path.join(os.getcwd(), "checkpoints/mlp_table_v2.jx")
# checkpoint_file = os.path.join(os.getcwd(), "checkpoints/mlp_table_v3_subset.jx")
checkpoint_file = os.path.join(os.getcwd(), "checkpoints/mlp_table_v3.jx")
checkpointer = ocp.StandardCheckpointer()
print(os.getcwd())

In [ ]:
_, params = nnx.split(model)
checkpointer.save(checkpoint_file, params, force=True)

In [ ]:
# abstract_model = nnx.eval_shape(lambda: model)
# graphdef, abstract_params = nnx.split(abstract_model)

# params = checkpointer.restore(checkpoint_file, abstract_params)
# model = nnx.merge(graphdef, params)

# tests

In [ ]:
X, _, Y, _ = data.get_offline_regression_data(
    train_dict, x_labels=x_labels, y_labels=y_labels, standardize_input=False, include_scale=True
)

In [ ]:
X[0,:10,:]

In [ ]:
X, _, Y, _ = data.get_offline_regression_data(
    train_dict, x_labels=x_labels, y_labels=y_labels, standardize_input=False, include_scale=False
)

In [ ]:
X.shape

In [ ]:
x_labels = ["rho", "fscalar", "vel_disp", "vel_div"]
# x_labels = ["rho", "fscalar"]
y_labels = ["P"]

X, _, Y, _ = data.get_offline_regression_data(
    train_dict, x_labels=x_labels, y_labels=y_labels, standardize_input=False, include_scale=True
)

print(X.shape)

In [ ]:
i = 0

cosmo = train_dict["cosmo"]
scales = train_dict["scales"]

dm_poss = train_dict["dm_poss"]
dm_vels = train_dict["dm_vels"]

gas_poss = train_dict["gas_poss"]
gas_vels = train_dict["gas_vels"]

scale = scales[i]
gas_pos = gas_poss[i]
gas_vel = gas_vels[i]

N_gas = cic_paint(jnp.zeros(mesh_shape), gas_pos)
rho_gas = N_gas * cosmo.Omega_b / (cosmo.Omega_c + cosmo.Omega_b)
gas_N = cic_read(N_gas, gas_pos)
gas_rho = cic_read(rho_gas, gas_pos)

X_online = data.get_hpm_inputs(
    scale,
    gas_pos,
    gas_vel,
    gas_rho,
    rho_gas,
    gas_N,
    mesh_shape,
    return_vel=True,
    # return_vel=False,
    return_field=False,
)

print(X_online.shape)

In [ ]:
for j in range(X.shape[-1]):
    print(jnp.allclose(X_online[:,j], X[i,:,j], rtol=1e-3, atol=1e-3))
    print(X_online[:,j])
    print(X[i,:,j], "\n")

In [ ]:
X_online[:,0]

In [ ]:
X[i,:,0]

In [ ]:
X